In [2]:
using LinearAlgebra, BenchmarkTools, PolynomialRoots, StaticArrays, DataStructures, Statistics

# Fully-split velocity Lagrangian PDMP (Technical WIP - not complete yet)
The main branch has the funcitonal code for the Split Lagrangian PDMP and Covariance Adaptive BPS. This part is under development.

## Mathemagical preliminaries
### A quick outline
Our goal shall be to define a set of exactly solvable problems to see that the fully-split velocity Lagrangian PDMP can be handled almost entirely analytically. First we establish what the rates and dynamics look like. Then we show that the rates are cubic in each velocity component, and that the dynamics are exactly solvable. We shall compute and store the corresponding coefficients of the dynamics ODE and rates as tensors. Then we shall implement a program to compute the rates and update rules for the split-velocity part of the PDMP. This will enable us to merge this code with the previously written PDMP code to finally run this in parallell to CA-BPS or SL-PDMP.


### Splitting equations and general rate solution
Consider, for simplicity, a general split PDMP with each transitions $\alpha \to \beta$, i.e. only the split dynamic changes (Other events are entirely possible to consider in the same framework, but clutter up notation, so let us leave them out for now).  Let $\lambda(\alpha \to \beta)$ be the associated rates. To preserve a distribution $\mu$ the split flow is taken to satisfy

$
-\mu^{-1}\text{div}_\alpha(\mu \Phi^\alpha) = \sum_\beta  \lambda(\alpha \to \beta) - \lambda(\alpha \to \beta)
$

and denoting the divergence (sign?) on the LHS by $A_\alpha$, and $\lambda_{\alpha\beta} =\lambda(\alpha \to \beta) $ we get

$
A_\alpha = \sum_\beta \lambda_{\alpha\beta}-\lambda_{\beta\alpha}
$

Of course, if the original $\Phi$ satisfies $\text{div}(\mu \Phi)=0$ then so does the sum of split flows, as the sum of divergences $A_\alpha$ vanish by linearity of the divergence operator.

In trying to define the rates we have to solve a constraint on an anti-symmetric matrix. Clearly, adding any symmetric part to $\lambda_{\alpha\beta}$ is inconsequential for the constraint above. Effectively, under such a transformation, the altered flow ${\alpha \to \beta}$ is offset by an equivalent flow $\beta \to \alpha$, and in simulation we will get more or fewer events but the overall divergence is not altered. Since events are typically costly we generally will want $\lambda_{{\alpha\beta}}$ to have as small a symmetric part as possible. Alas, $\lambda_{{\alpha\beta}} \geq 0 $ so the symmetric part cannot be zero identically.

Rather than solving the above positivity and anti-symmetry constraints on $\lambda_{{\alpha\beta}}$ we can assume 

$
\lambda_{{\alpha\beta}} = [\rho_{\alpha\beta}]^+
$

where $\rho_{\alpha\beta}$ is some anti-symmetric matrix. Then 

$
\lambda_{\alpha\beta} - \lambda_{\beta\alpha}= [\rho_{\alpha\beta}]^+ - [\rho_{\beta\alpha}]^+ = [\rho_{\alpha\beta}]^+-[-\rho_{\alpha\beta}]^+ = \rho_{\alpha \beta}
$

and thus we have to solve

$
A_\alpha = \sum_\beta \rho_{\alpha\beta}
$

where the $A_\alpha$ are determined by $-\mu^{-1}\text{div}_\alpha(\mu \Phi_\alpha)$. A simple solution is given by

$
\rho_{\alpha \beta} = (A^\alpha-A^\beta)/n
$

where $n$ is the number of split states. This solution gives us a direct interpretation of the rates: it is possible to transition into a state $\beta$ _precisely_ when the divergence of flow associated to $\beta$ exceeds the divergence of the flow in the current state $\alpha$. The greater the difference in divergence, the more likely we are to transition into the given state.


Note that the $A_\alpha$ are arbitrary (but must satisfy $\sum_\alpha A_\alpha=0$), and so this is a valid solution and rates for any split. If for some reason we want to have a substantially increased rate we may adjust the rate matrix $\lambda_{\alpha\beta} \to \lambda_{\alpha\beta} + R_{\alpha\beta}$ where $R_{\alpha\beta}$ is any positive definite symmetric matrix. 

As we shall soon see there are several other solutions of interest.

### Preferential splitting rates
Suppose we are interested in a particular state, say, $\alpha = 0$, for which we want to have as small rates $\lambda_{0\beta}$ as possible. In other words, we would like 'stay' in the state $\alpha = 0$ for longer (or, at the very least, have fewer events $0 \to \beta$). Then, as we shall see, the above rates are not ideal. The rate $\lambda_{0 \to \text{any}} = \sum_\beta \lambda_{0\beta}$
becomes

$\lambda_{0 \to \text{any}} = \frac{1}{n+1}\sum_\beta [A_0-A_\beta]^+\geq \frac{1}{n+1}[\sum_\beta A_0-A_\beta]^+ = \frac{1}{n+1}[\sum_\beta A_0]^+ = [A_0]^+$

with equality if and only if $A_0 - A_\beta \geq 0$ for every $\beta$.

Had we only split into $0$ and taken the other splits $\beta= 1,2,\ldots, n $ as a single state $I$ the above 'recipe' would instead declare

$\lambda_{0 \to I} = \frac{1}{2}[A_0 - A_I]^+ =[A_0]^+$

since $A_0 + A_I = 0$. Clearly the former choice means that we transition into $I$ with a frequency that is *at least* as big as that of the latter choice, but generally larger.

Thus, if possible, we would like to use another rate for transitions $0 \to \beta$. Mercifully there are some obvious choices. Considering the equation

$A_0 = \sum_\beta \lambda_{0\beta}-\lambda_{\beta 0 }$

we can, as above, pick an anti-symmetric $\rho$-matrix and $\lambda_{0 \beta} = [\rho_{0\beta}]^+$ and $\lambda_{0 \beta} = [-\rho_{0\beta}]^+$  which leads to

$A_0 = \sum_\beta \rho_{0\beta}$

which we can solve by letting $\rho_{0\beta} = \frac{1}{n} A_0 = -\rho_{\beta 0}$ for $\beta \neq 0$. How should we interpret this choice for the rates? Now we only shift from $\alpha =0$ to _some_ (note that we effectively pick which uniformly at random) $\beta$ if the average divergence in $-A_\beta = A_0$ exceeds $0$, rather than if some individual divergence does so.

This has some consequence for the divergence equations for other splits. Let lower-case latin letters $a \neq 0$ index the $n$ splits other than $\alpha = 0$. We can in fact apply the same recipe as above again. To see this we have

$A_a = \sum_\alpha \lambda_{a\alpha}-\lambda_{\alpha a} =  (\lambda_{a0}-\lambda_{0a}) + \sum_b\lambda_{ab}-\lambda_{b a}  = \rho_{a0} + \sum_b\lambda_{ab}-\lambda_{b a} 
=-\frac{A_0}{n} + \sum_b\rho_{ab}$

Thus, picking $\rho_{ab} = \frac{1}{n}(A_a - A_b)$ we get

$ \sum_b\rho_{ab} = A_a - \frac{1}{n}\sum_b A_b = A_a  - \frac{1}{n}(-A_0 + \sum_{\beta} A_\beta ) = A_a + \frac{1}{n}A_0$

since $\sum_b A_b = -A_0 + \sum_\beta A_\beta = -A_0$ due to the vanishing overall divergence. Hence the $A_0$ terms in $a$-divergence equation cancel and the solution

$\rho_{ab} = \frac{1}{n}(A_a - A_b)$ 

is satisfactory.

### The Lagrangian split-velocity case
We now turn our attention to a splitting of the Lagrangian dynamics, so that we have one state corresponding to evolution of position, and one state for each evolution of a velocity component $v^i$. Thus, in sampling in $\mathbb{R}^n$ we shall have $n+1$ split states. In our sampling we shall treat position updates preferentially (
Shifting between velocities is *hopefully* inexpensive, but shifting between position and velocity updates is quite costly as it incurs a computation of third order derivatives.).

#### Notation, notation, my kingdom for better notation
We let the position flow correspond to the split state variable $\alpha = n+1$, so that the $I$:th velocity component can be chosen to be represented by $\alpha = I$, and so that matrix-enumerations match this (Matrices and arrays in Julia are, of course - as with any other sane language - indexed starting from 1.). We shall, in what comes, adopt a somewhat specialized notation. Namely, we let capital latin latters (typically $I,J,K,\ldots$) denote indices over a single dimension (the splitting dimension) which means we _**do not apply the Einstein summation convention**_ to these indices. By lower-case latin letters we refer to components that run across 'all dimensions other than the one currently evolving'. That is to say, if e.g. $\alpha = 3$ then a lower case letter $a$ tracks across all indices $1, 2, 4, 5, \ldots, n$. Greek letters refer always to the full set of indices $1,2,3,4,\ldots, n$. So if $\alpha = I$ we have

$\Gamma^\alpha_{\alpha \beta} = \Gamma^I_{I\beta} + \Gamma^{a}_{a\beta}$

but sometimes we encounter expressions like 

$\Gamma^\alpha_{J\beta}v^Jv^\beta = \Gamma^\alpha_{JI}v^Jv^I  + \Gamma^\alpha_{Ja}v^Jv^a$

where for $J\neq I$ the $a$ now does run over $J$, since $I$ is assumed to be evolving. Very tricky.

#### The equation of motion and divergences
The Lagrangian dynamics are described at length in the paper. Let us summarize some key features:

The flow in position, now associated to the $n+1$:th split state, is, componentwise:

$(\Phi_{n+1})^a = v^a$ 

and for the flow in the $I$:the velocity component we have (in a very slight abuse of notation we take $\Phi_I^I = \Phi^I$)

$\Phi^I = (-\eta - G^{-1}\nabla \phi)^I = -\Gamma^I_{\alpha\beta}v^\alpha v^\beta - G^{I\alpha}\partial_\alpha \phi = $

$= -\Gamma^I_{II} (v^I)^2 - 2\Gamma^I_{Ia}v^Iv^a-\Gamma^I_{ab}v^av^b-G^{I\alpha}\partial_\alpha \phi $

and since we do not (primarily) deal with BPS-type events we set $\phi = -\log \pi + \frac{1}{2}\log \det G$. However, since this is independent of $v$ it does not change the qualitative nature of the ODE defined by the flow along $\Phi_{I}$. We can express the flow in $v^J$ as a form of the _Ricatti equation_:

$du/dt = au^2 + b u + c $

for coefficients depending on $J$:

$ a_{;J} = -\Gamma^J_{JJ}$

$ b_{;J} = - 2\Gamma^J_{Ja}v^a$

$ c_{;J} = -G^{J\alpha}\partial_\alpha \phi-\Gamma^J_{ab}v^av^b$

The divergences for the velocities are straightforward to compute, but less pleasant to expand in our specialized indices. We shall need an expression for each divergence $A_I$ as a function of the evolving velocity component $u = v^J$. The divergences are cubic in the velocities so we shall expand $A_{I;J}(u) = A_{I;J}^{(0)} + A_{I;J}^{(1)}u^1+A_{I;J}^{(2)}u^2+A_{I;J}^{(3)}u^3 $ 

$A_I = -\mu^{-1}\text{div}_I(\mu \Phi_I) = (2\Gamma^I_{I\alpha} + \Phi^IG_{I\alpha})v^\alpha = 
2\Gamma^I_{I\alpha}v^\alpha - \Gamma^I_{\mu\nu}G_{I\alpha}v^\mu v^\nu v^\alpha - G^{I\beta} G_{I\alpha}(\partial_\beta \phi)v^\alpha $

and hence

$A_{I;J}^{(0)} = 2\Gamma^I_{Ia}v^a - \Gamma^I_{ab}G_{Ic}v^a v^b v^c - G^{I\beta} G_{Ia}(\partial_\beta \phi)v^a$

$A_{I;J}^{(1)} = 2\Gamma^I_{IJ} - 2\Gamma^I_{aJ}G_{Ib}v^a  v^b - \Gamma^I_{ab}G_{IJ}v^a v^b  - G^{I\beta} G_{IJ}(\partial_\beta \phi) $

$A_{I;J}^{(2)} = -\Gamma^I_{JJ} G_{Ia}v^a - 2\Gamma^I_{Ja}G_{IJ}v^a  $

$A_{I;J}^{(3)} = -\Gamma^I_{JJ}G_{IJ}$

The final divergence is

$A_{n+1; J} = -2\Gamma^\alpha_{\alpha \beta}v^\beta + \Gamma^\mu_{\alpha \beta} G_{\mu \nu} v^\alpha v^\beta v^\nu + (\partial_\alpha \phi)v^\alpha$

so (as might be inferred also from the above expression, and knowing that $A_{n+1} = \mu^{-1}\text{div}_x(\mu \Phi) = -\mu^{-1}\text{div}_v(\mu \Phi) = \sum_a A_a$)

$A_{n+1; J}^{(0)}  = -2\Gamma^\alpha_{\alpha a}v^a + \Gamma^\mu_{ab} G_{\mu c} v^a v^b v^c + (\partial_a \phi)v^a$

$A_{n+1; J}^{(1)}  = -2\Gamma^\alpha_{\alpha J} + 2\Gamma^\mu_{J a} G_{\mu b} v^a v^b + \Gamma^\mu_{ab } G_{\mu J} v^a v^b + (\partial_J \phi)$

$A_{n+1; J}^{(2)}  =  2\Gamma^\mu_{J a} G_{\mu J} v^a + \Gamma^\mu_{JJ}G_{\mu a}$

$A_{n+1; J}^{(3)}  =  \Gamma^\mu_{J J} G_{\mu J} $

Of course, to compute $A_{n+1}$ we should certainly use the form of the $A_a$ above. Since, at all times, we are only interested in a single $J$ at a time, and the relationships between $A_{I;J}$ and $A_{I;K}$ are somewhat complicated, we do not benefit greatly from computing the full $A_{I;J}^{(n)}$.

## Implementing the flow, divergences, rates etc

In [3]:
#Computing the object A^{(n)}_{I;J}:
function compute_divergences!(A, reduced_v, dim, J,  Γ, G, G_inv, ∇φ, vel)
    reduced_v .= vel
    reduced_v[J] = 0.0
    
    @views A[dim+1, :] .= 0.0

    @inbounds for I in 1:dim
        @views A[I, 4] = -Γ[I, J, J] * G[I, J]
        
        A[dim+1, 4] -= A[I, 4]

        @views A[I, 3] = -Γ[I, J, J] * dot(G[I, :], reduced_v) - 2 * G[I, J] * dot(Γ[I, J, :], reduced_v)

        A[dim+1, 3] -= A[I, 3]

        @views A[I, 2] = 2*Γ[I, I, J] - 2*dot(Γ[I, J, :], reduced_v)*dot(G[I, :], reduced_v) - G[I, J] * dot(reduced_v, Γ[I, :, :], reduced_v) - G[I, J] * dot(G_inv[I, :], ∇φ)

        A[dim+1, 2] -= A[I, 2]

        @views A[I, 1] = 2*dot(Γ[I, I, :], reduced_v) - dot(reduced_v, Γ[I, :, :], reduced_v)*dot(G[I, :], reduced_v) - dot(G[I, :], reduced_v) * dot(G_inv[I, :], ∇φ)

        A[dim+1, 1] -= A[I, 1]
    end

    return A
end        

compute_divergences! (generic function with 1 method)

As a rough order of magnitude estimate this should take something like $\sim 10 \mu s$ to compute for a 20-dimensional space.

#### $J\to I$ and $J\to 0$ rate under $J$-flow from the divergences
We have $\rho_{JI} = (A_{J;J}-A_{I;J})/n $ and $\rho_{J , n+1} = -A_{n+1;J}/n$ so given the above we can compute the rates with ease. Since we can solve each flow (almost) exactly we shall not have need of the explicit rates, but rather the $\rho_{JI}^{(n)}$

In [4]:
function compute_rho!(ρJ, J, A, dim)
    @inbounds for I in 1:dim
        if I ≠ J
            @views ρJ[I,:] .= A[J,:] - A[I,:]
        else
            @views ρJ[J,:] .= 0.0
        end
    end

    @views ρJ[n+1, :] .= (-A[n+1,:] ./ dim) 

    return ρJ
end

compute_rho! (generic function with 1 method)

### Ricatti equation and the velocity flow
The fully split velocity satisfies the Ricatti equation

$du/dt = a u^2 + b u + c$

for some $a,b,c$. This admits the solution

$u(t) = (\kappa \tan(\kappa(t+t_0))-(b/2))/a$

where $\kappa = \sqrt{4ac-b^2}/2$

Notably, if $\kappa$ is imaginary $\kappa = i k$ for some $k \in \mathbb{R}$ then

$\kappa \tan(\kappa s) = i k \tan( iks ) = -k\text{tanh}(ks)$

There are, however, numerous edge cases that must be handled. These cause a substantial headache as they must be integrated (for exact rate integrals) to the $n$:th power for $n=1,2,3$.

If $a \neq 0$ then we can transform into a reduced form by letting $y = u a$ and we get a reduced equation $y' = y^2 + b y + ca$ where $y_0 = a u_0$

### The 6 solutions of the Ricatti equation
We shall have to include Ricatti-equation tests, comparing $u'(t)$ with $au^2+bu+c$ for all interesting variable combinations. Above we noted that we essentially had 6 distinct solutions (which I've enumerated in the ~Pythonic~ way for who knows what reason):

0) Stationary solutions for which $u = u_0$
1) (Non-stationary) Solutions for $a = b= 0, c\neq 0$, leading to $u(t) = c t + u_0$
2) (Non-stationary) Solutions for $a = 0, b\neq 0, c\neq 0$, leading to $u(t) = (u_0 + p)e^{bt} - p$ where $p = c/b$
3) (Non-stationary) Solutions with $a\neq 0$ but $\kappa^2 = 4ac -b^2 =0$ (and so $c = b^2/(4a)$) for which 

    $u(t) =  \frac{1}{a}(\frac{-b}{2} + \frac{l_0}{1 - l_0t})$ where $l_0 =  (\frac{b}{2} + au_0)$.
    
4) (Non-stationary) Solutions with $a\neq 0$ and $\kappa^2 = ac -(b^2/4) > 0$. Then 

    $u(t) = a^{-1}(\kappa  \tan(s_0 + (\kappa t)) - q) $ where $q =  b/2$ and $s_0$ is such that $u(0)=u_0$.

5) (Non-stationary) Solutions with $a\neq 0$ and $\kappa^2 = ac -(b^2/4) < 0$. Then 

    $u(t) = a^{-1}(-\frac{b}{2} + \tilde\kappa(-1 + 2(1-\tilde\omega_0e^{2\tilde\kappa t})^{-1})) $ where $\tilde\kappa = \sqrt{b^2/4 - ac}$ and $\tilde\omega_0 = 1 - \frac{2\tilde\kappa}{u_0 a + b/2 + \tilde\kappa}$. Note that $u_0 \neq -\frac{b}{2} + \tilde \kappa$ since we assume the solution isn't stationary.

These solutions have one thing in common (the first case could arguably be excluded): The sign of $du/dt$ only depends on some constant in each case - thus it is simple to see what range of future $u$ we have, given $u_0$ and this determining parameter. We have

1) $c$ since $u'(t) = c$ 
2) $b$ since $u'(t) = (pb+u_0b) e^{bt} = (c+u_0b)e^{bt}$ which has the sign of $c + u_0b$
3) $a$ since $u'(t) = a^{-1}l_0^2(1-l_0t)^{-2}$ which has the sign of $a$
4) $a$ since $u'(t) = a^{-1} \kappa^2 (1+ \tan^2(s_0+\kappa t))$ which has the sign of $a$
5) $a$ since $u'(t) = 2a^{-1}\tilde\kappa^2 \omega_0 e^{2\tilde\kappa t} (1-\omega_0\exp(2\tilde\kappa t))^{-2}$ which has the sign of $a\omega_0$

In [23]:
#There are a lot of exact float checks here. This has to be managed/studied to see where issues arise.
function reduced_ricatti_solution(t, β, γ; y0 = 0.0)
    if isapprox(4*γ, β^2)
        l0 = y0+(β/2)
        return (l0*inv(1-(l0*t)))-(β/2)  #Type 3 velocity
    end 
    κ2 = γ - ((β^2)/4)
    k = sqrt(abs(κ2))
    δ = β/2 
    if κ2 > 0
        s0 = atan((y0 + δ)/k)# t0 = s0/k
        return (k * tan(s0 + (k*t))) - δ #Type 4 velocity
    else 
        ω_0 = 1 - (2*k/(y0 + δ + k)) 
        return -δ + (k*(-1 + (2*inv(1 - ω_0*exp(2*k*t))))) #Type 5 velocity
    end
end

function ricatti_solution(t,a,b,c; u0 = 0.0, verbose = false)
    if isapprox(a*(u0^2) + (b*u0) , -c)
        verbose ? (@warn "Stationary velocity, rates are constant") : nothing
        return u0 #Type 0 velocity
    end
    if a ≈ 0 
        if b ≈ 0 
            #Cant have c=0 in this case as that would trigger the above constraint. Of course, in really dicey accuracy questions this might be relevant.
            if c ≈ 0
                verbose ? (@warn "Stationary velocity, rates are constant") : nothing
                return u0 #Type 0 velocity
            end
            return c*t + u0 #Type 1 velocity
        end
        q = c/b
        return (u0 + q)*exp(b*t) - q #Type 2 velocity
    end
    return reduced_ricatti_solution(t, b, c*a; y0 = a*u0)/a
end

ricatti_solution (generic function with 1 method)

### Tests

In [6]:
function rel_failure(v_approx, v)
    if v ≈ v_approx
        return 0.0
    end
    if iszero(v)
        if iszero(v_approx)
            return 0.0
        end
        return 1.0
    end
    return abs(v_approx - v)/(abs(v_approx) + abs(v))
end

rel_failure (generic function with 1 method)

#### Test of solution 0

In [7]:
function test_0(;t_range = 0.0:0.001: 3.0)
    a = (10^rand())*randn()
    b = (10^rand())*randn()
    c = (b^2)/4
    u0 = (10^rand())*randn()
    c = - (a*(u0^2)) - (b*u0)
    u_expected(t) = u0
    u(t) = ricatti_solution(t, a,b,c, u0 = u0, verbose = false)
    du_expected(t) = 0.0
    du(t) = (a*(u(t)^2)) + (b * u(t)) + c
    failure_in_value = 0.0
    failure_in_der = 0.0
    n = 0
    for t in t_range
        failure_in_value += rel_failure(u(t), u_expected(t))
        failure_in_der += rel_failure(du(t), du_expected(t))
        n+=1
    end
    if failure_in_der ≠ 0 || failure_in_value ≠ 0
        println("Total relative failure from expected solution: $failure_in_value \n Total relative failure in derivative $failure_in_der \n (over $n samples)")
    end
    nothing
end

test_0 (generic function with 1 method)

In [8]:
for i = 1:100
    test_0()
end

#### Test of solution 1

In [9]:
function test_1(;t_range = 0.0:0.001: 3.0)
    a = 0.0
    b = 0.0
    c = (10^rand())*randn()
    while c ≈ 0
        c = (10^rand())*randn()
    end
    u0 = (10^rand())*randn()
    u_expected(t) = c*t + u0
    u(t) = ricatti_solution(t, a,b,c, u0 = u0, verbose = false)
    du_expected(t) = c
    du(t) = (a*(u(t)^2)) + (b * u(t)) + c
    failure_in_value = 0.0
    failure_in_der = 0.0
    n = 0
    for t in t_range
        failure_in_value += rel_failure(u(t), u_expected(t))
        failure_in_der += rel_failure(du(t), du_expected(t))
        n+=1
    end
    if failure_in_der ≠ 0 || failure_in_value ≠ 0
        println("Total relative failure from expected solution: $failure_in_value \n Total relative failure in derivative $failure_in_der \n (over $n samples)")
    end
    nothing
end

test_1 (generic function with 1 method)

In [10]:
for i = 1:100
    test_1()
end

#### Test of solution 2

In [11]:
function test_2(;t_range = 0.0:0.0001: 3.0, threshold = 10^(-16))
    a = 0.0
    b = (10^rand())*randn()

    while b ≈ 0
        b = (10^rand())*randn()
    end

    c = (10^rand())*randn()
    while c ≈ 0
        c = (10^rand())*randn()
    end

    u0 = (10^rand())*randn()
    while (b*u0) + c ≈ 0 || c ≈ 0
        c = (10^rand())*randn()
    end

    p = c/b
    u_expected(t) = (u0 + p)*exp(b*t) - p
    u(t) = ricatti_solution(t, a,b,c, u0 = u0, verbose = false)
    du_expected(t) = (u0 + p)*b*exp(b*t)
    du(t) = (a*(u(t)^2)) + (b * u(t)) + c
    
    n=0
    rel_val_failure = Float64[]
    rel_der_failure = Float64[]
    for t in t_range
        push!(rel_val_failure, rel_failure(u(t), u_expected(t)))
        push!(rel_der_failure, rel_failure(du(t), du_expected(t)))
        n+=1
    end
    failure_in_value = sum(rel_val_failure)
    failure_in_der = sum(rel_der_failure)
    
    mv = maximum(rel_val_failure)
    md = maximum(rel_der_failure)
   
    
    if mv > threshold || md > threshold 
        if failure_in_value ≠ 0
            println("Failure in val! Total relative failure from expected solution: $failure_in_value")
            ind = findfirst(x-> x==mv, rel_val_failure) #I know, inefficient
            t= t_range[ind]
            μval = mean(rel_val_failure)
            σval = std(rel_val_failure)
            println("Failure in val \n Maximal failure: $mv at t = $t\n Mean value failure: $μval \n Standard deviation: $σval \n Value: $((u(t), u_expected(t)))")
        end

        if failure_in_der ≠ 0
            println("Failure in der! Total relative failure from expected solution: $failure_in_der")
            ind = findfirst(x-> x==md, rel_der_failure) #I know, inefficient
            t= t_range[ind]
            μder = mean(rel_der_failure)
            σder = std(rel_der_failure)
            println("Maximal failure: $md at t = $t\n Mean der failure: $μder \n Standard deviation: $σder \n Value: $((du(t), du_expected(t)))")
        end
        println("Coefficients: (a,b,c, u0) = $((a,b,c,u0))")
    end
    nothing
end

test_2 (generic function with 1 method)

In [68]:
for i = 1:100
    test_2()
end

Failure in der! Total relative failure from expected solution: 1.7286097666261173e-5
Maximal failure: 5.964071398320367e-8 at t = 2.9953
 Mean der failure: 5.761840494070589e-10 
 Standard deviation: 3.443363380485409e-9 
 Value: (2.9102309451189967e-10, 2.9102312922555203e-10)
Coefficients: (a,b,c, u0) = (0.0, -8.305373916538784, 0.16295766321323701, -2.2116368153299306)
Failure in der! Total relative failure from expected solution: 1.561068088113283e-5
Maximal failure: 5.622526092083952e-8 at t = 2.9936
 Mean der failure: 5.203386847482694e-10 
 Standard deviation: 3.2425926944606846e-9 
 Value: (1.1189024151647686e-9, 1.1189022893436152e-9)
Coefficients: (a,b,c, u0) = (0.0, -8.096193703151506, 0.9752495647421582, -4.518332883806209)
Failure in der! Total relative failure from expected solution: 1.2319863900689026e-5
Maximal failure: 4.668773321446419e-8 at t = 2.9954
 Mean der failure: 4.1064844174157615e-10 
 Standard deviation: 2.6817853277141556e-9 
 Value: (-1.258557591476972e-9

Here the derivative fails in many tests. Fundamentally what appears to be the issue is that large or small exponentials are added to a float of some *reasonable* size. It is generally a difficult issue that we shall not attempt to overcome.

Up to float-issues, the test appears to pass - and since our only issues are with small differences of the derivative we feel certain about the validity of the solution. Note that the derivatives are not explicitly used in the computations ahead - they are present here as a sanity check on our "solutions" as actual solutions of the ODE.

#### Test of solution 3

$u(t) =  \frac{1}{a}(\frac{-b}{2} + \frac{l_0}{1 - l_0t})$ where $l_0 =  (\frac{b}{2} + au_0)$

In [12]:
function test_3(;t_range = 0.0:0.0001: 3.0, threshold = 10^(-15))
    a = (10^rand())*randn()
    while a ≈ 0
        a = (10^rand())*randn()
    end
    b = (10^rand())*randn()
    c = (b^2)/(4*a)

    u0 = (10^rand())*randn()
    while a*(u0^2) + b*(u0) + c ≈ 0 
        u0 = (10^rand())*randn()
    end

    l_0 = (a*u0) + (b/2)
    u_expected(t) = ((-b/2) + (l_0*inv(1 - (l_0*t))))*inv(a)
    u(t) = ricatti_solution(t, a,b,c, u0 = u0, verbose = false)
    du_expected(t) = l_0^2*((inv(1 - (l_0*t)))^2)/a
    du(t) = (a*(u(t)^2)) + (b * u(t)) + c
    
    n=0
    rel_val_failure = Float64[]
    rel_der_failure = Float64[]
    for t in t_range
        push!(rel_val_failure, rel_failure(u(t), u_expected(t)))
        push!(rel_der_failure, rel_failure(du(t), du_expected(t)))
        n+=1
    end
    failure_in_value = sum(rel_val_failure)
    failure_in_der = sum(rel_der_failure)
    
    mv = maximum(rel_val_failure)
    md = maximum(rel_der_failure)
   
    if mv > threshold || md > threshold 
        if failure_in_value ≠ 0
            println("Failure in val! Total relative failure from expected solution: $failure_in_value")
            ind = findfirst(x-> x==mv, rel_val_failure) #I know, inefficient
            t= t_range[ind]
            μval = mean(rel_val_failure)
            σval = std(rel_val_failure)
            println("Failure in val \n Maximal failure: $mv at t = $t\n Mean value failure: $μval \n Standard deviation: $σval \n Value: $((u(t), u_expected(t)))")
        end

        if failure_in_der ≠ 0
            println("Failure in der! Total relative failure from expected solution: $failure_in_der")
            ind = findfirst(x-> x==md, rel_der_failure) #I know, inefficient
            t= t_range[ind]
            μder = mean(rel_der_failure)
            σder = std(rel_der_failure)
            println("Maximal failure: $md at t = $t\n Mean der failure: $μder \n Standard deviation: $σder \n Value: $((du(t), du_expected(t)))")
        end
        println("Coefficients: (a,b,c, u0) = $((a,b,c,u0))")
    end
    nothing
end

test_3 (generic function with 1 method)

In [13]:
for i=1:100
    test_3()
end

#### Test of solution 4
$u(t) = a^{-1}(\kappa  \tan(s_0 + (\kappa t)) - q) $ where $q =  b/2$ and $s_0$ is such that $u(0)=u_0$, $\kappa^2 = ac - b^2/2$ 

In [14]:
function test_4(;t_range = 0.0:0.0001: 3.0, threshold = 10^(-15))
    a = (10^rand())*randn()
    while a ≈ 0
        a = (10^rand())*randn()
    end
    b = (10^rand())*randn()
    
    κ2 = (10^rand())*(randn()^2) 
    while κ2 ≈ 0
        κ2 = (10^rand())*(randn()^2)
    end

    c = (κ2 + ((b^2)/4))/a

    u0 = (10^rand())*randn()
    while a*(u0^2) + b*(u0) + c ≈ 0 
        u0 = (10^rand())*randn()
    end

    κ = sqrt(κ2)
    s0 = atan((a*u0 + (b/2))/κ)


    u_expected(t) = inv(a)*(-(b/2) + κ*tan(s0 + (κ*t)))
    u(t) = ricatti_solution(t, a,b,c, u0 = u0, verbose = false)
    du_expected(t) = (κ^2)*inv(a)*(1 + (tan(s0 + (κ*t))^2))
    du(t) = (a*(u(t)^2)) + (b * u(t)) + c
    
    n=0
    rel_val_failure = Float64[]
    rel_der_failure = Float64[]
    for t in t_range
        push!(rel_val_failure, rel_failure(u(t), u_expected(t)))
        push!(rel_der_failure, rel_failure(du(t), du_expected(t)))
        n+=1
    end
    failure_in_value = sum(rel_val_failure)
    failure_in_der = sum(rel_der_failure)
    
    mv = maximum(rel_val_failure)
    md = maximum(rel_der_failure)
   
    if mv > threshold || md > threshold 
        if failure_in_value ≠ 0
            println("Failure in val! Total relative failure from expected solution: $failure_in_value")
            ind = findfirst(x-> x==mv, rel_val_failure) #I know, inefficient
            t= t_range[ind]
            μval = mean(rel_val_failure)
            σval = std(rel_val_failure)
            println("Failure in val \n Maximal failure: $mv at t = $t\n Mean value failure: $μval \n Standard deviation: $σval \n Value: $((u(t), u_expected(t)))")
        end

        if failure_in_der ≠ 0
            println("Failure in der! Total relative failure from expected solution: $failure_in_der")
            ind = findfirst(x-> x==md, rel_der_failure) #I know, inefficient
            t= t_range[ind]
            μder = mean(rel_der_failure)
            σder = std(rel_der_failure)
            println("Maximal failure: $md at t = $t\n Mean der failure: $μder \n Standard deviation: $σder \n Value: $((du(t), du_expected(t)))")
        end
        println("Coefficients: (a,b,c, u0) = $((a,b,c,u0))")
    end
    nothing
end

test_4 (generic function with 1 method)

In [15]:
for i = 1:100
    test_4()
end

Failure in der! Total relative failure from expected solution: 7.636973125215023e-9
Maximal failure: 7.636973125215023e-9 at t = 2.4531
 Mean der failure: 2.5455728559764754e-13 
 Standard deviation: 4.4091350046657947e-11 
 Value: (5.977698322868472e13, 5.97769823156543e13)
Coefficients: (a,b,c, u0) = (1.843689334671619, -5.5629783786686415, 7.430393151787118, -2.880385959819399)


This appears generally stable.

#### Test of solution 5


$u(t) = a^{-1}(-\frac{b}{2} + \tilde\kappa(-1 + 2(1-\tilde\omega_0e^{2\tilde\kappa t})^{-1})) $ where $\tilde\kappa = \sqrt{b^2/4 - ac}$ and $\tilde\omega_0 = 1 - \frac{2\tilde\kappa}{u_0 a + b/2 + \tilde\kappa}$. Note that $u_0 \neq -\frac{b}{2} + \tilde \kappa$ since we assume the solution isn't stationary.

In [37]:
function test_5(;t_range = 0.0:0.0001: 3.0, threshold = 10^(-15))
    a = (10^rand())*randn()
    while a ≈ 0
        a = (10^rand())*randn()
    end
    b = (10^rand())*randn()
    
    κ2 = -(10^rand())*(randn()^2) 
    while κ2 ≈ 0
        κ2 = -(10^rand())*(randn()^2)
    end

    c = (κ2 + ((b^2)/4))/a

    u0 = (10^rand())*randn()
    while a*(u0^2) + b*(u0) + c ≈ 0 
        u0 = (10^rand())*randn()
    end

    κ = sqrt(abs(κ2))
    ω0 = 1- (2*κ*inv((u0*a) + (b/2) + κ))


    u_expected(t) = inv(a)*(-(b/2) + κ*(-1 + (2*inv(1-ω0*exp(2*κ*t)))))
    u(t) = ricatti_solution(t, a,b,c, u0 = u0, verbose = false)
    du_expected(t) = ((2*κ)^2)*inv(a)*ω0*((inv(1 - ω0*exp(2*κ*t)))^2)*exp(2*κ*t)
    du(t) = (a*(u(t)^2)) + (b * u(t)) + c
    
    n=0
    rel_val_failure = Float64[]
    rel_der_failure = Float64[]
    for t in t_range
        push!(rel_val_failure, rel_failure(u(t), u_expected(t)))
        push!(rel_der_failure, rel_failure(du(t), du_expected(t)))
        n+=1
    end
    failure_in_value = sum(rel_val_failure)
    failure_in_der = sum(rel_der_failure)
    
    mv = maximum(rel_val_failure)
    md = maximum(rel_der_failure)
   
    if mv > threshold || md > threshold 
        if failure_in_value ≠ 0
            println("Failure in val! Total relative failure from expected solution: $failure_in_value")
            ind = findfirst(x-> x==mv, rel_val_failure) #I know, inefficient
            t= t_range[ind]
            μval = mean(rel_val_failure)
            σval = std(rel_val_failure)
            println("Failure in val \n Maximal failure: $mv at t = $t\n Mean value failure: $μval \n Standard deviation: $σval \n Value: $((u(t), u_expected(t)))")
        end

        if failure_in_der ≠ 0
            println("Failure in der! Total relative failure from expected solution: $failure_in_der")
            ind = findfirst(x-> x==md, rel_der_failure) #I know, inefficient
            t= t_range[ind]
            μder = mean(rel_der_failure)
            σder = std(rel_der_failure)
            println("Maximal failure: $md at t = $t\n Mean der failure: $μder \n Standard deviation: $σder \n Value: $((du(t), du_expected(t)))")
        end
        println("Coefficients: (a,b,c, u0) = $((a,b,c,u0))")
    end
    nothing
end

test_5 (generic function with 1 method)

In [39]:
for i=1:100
    test_5()
end

Failure in der! Total relative failure from expected solution: 0.03824304035502652
Maximal failure: 0.00012574539751322128 at t = 2.9958
 Mean der failure: 1.274725520983518e-6 
 Standard deviation: 5.731872320057203e-6 
 Value: (2.4584778657299466e-11, 2.4578596589136012e-11)
Coefficients: (a,b,c, u0) = (2.5764104034661743, 2.0247106258449956, -7.340198912961346, -3.8320539287549833)
Failure in der! Total relative failure from expected solution: 4.489566507914064e-7
Maximal failure: 1.2854072558298939e-8 at t = 2.978
 Mean der failure: 1.4964722868951248e-11 
 Standard deviation: 3.6981005994707755e-10 
 Value: (-1.8861278405779558e-7, -1.8861277920891082e-7)
Coefficients: (a,b,c, u0) = (1.19342324636065, 0.9098213574242696, -7.311303683506376, -1.6176134976671195)
Failure in der! Total relative failure from expected solution: 0.0003563998355917567
Maximal failure: 8.661971665883391e-7 at t = 2.9793
 Mean der failure: 1.1879598533107453e-8 
 Standard deviation: 4.805804983392205e-8 
 

## Exact methods for rate integrals
### Exact methods
The rates are given by some expression

$\lambda^{IJ} = [\rho^{IJ}]^+ = [A^I - A^J]^+/n$

and the total rate in state $I$ is simply the sum $\lambda^I = \sum_J\lambda^{IJ}$. As discussed above the $A^I$ are cubic in $u(t)$. We can explicitly integrate each $\lambda^{IJ}$ *if* we know that it is positive. Thus, for a given $I$, we solve $\rho^{IJ}(u) = 0$ for all $J$ and order the individual solutions $u_1, u_2, \ldots$ such that $u_i < u_{i+1}$ if $du/dt > 0$, and $u_i> u_{i+1}$ else (Note that $du/dt$ is actually identically positive or negative for all $t$ we will consider, since $u(t)$ 'blows up' in finite time (this is a weird argument - it is true because we know that $u(t)$ has to blow up in finite time, and has to be a solution of the Ricatti equation)). This partitions the velocity space $U$ into sets over which the signs of all $\rho^{IJ}$ are constant and over these the sum $\sum_J \lambda^{IJ}$ can thus be computed.

In [5]:
pol = [9,3,-7,1]
@benchmark roots($pol)

BenchmarkTools.Trial: 10000 samples with 193 evaluations per sample.
 Range (min … max):  488.601 ns … 191.056 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     530.052 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   739.673 ns ±   2.742 μs  ┊ GC (mean ± σ):  7.05% ± 5.36%

  ▆█▅▅▃▄▃▂▁▁▁▁▁▂▂▂                                              ▁
  ███████████████████████▇▇▇▆▆▆▆▆▆▆▅▄▃▅▅▄▄▃▄▃▃▅▄▃▄▄▃▃▄▃▃▄▂▃▄▃▂▄ █
  489 ns        Histogram: log(frequency) by time       2.36 μs <

 Memory estimate: 496 bytes, allocs estimate: 8.

In [ ]:
"""
    cubic_real_roots(b, c, d; verbose=false)::Tuple

Finds the real roots of the cubic polynomial x^3 + b x^2 +c x + d.
"""
function cubic_real_roots(b, c, d; verbose=false)::Tuple
    Δ = b^2 - 3*c
    μ = (2*(b^3)) - (9*b*c) + (27*d)

    if iszero(Δ) && iszero(μ) #Should check vs threshold.
        return (-b/3,) 
    end

    #Optimize: Add special statement for when Δ = 0 (within threshold.)
    L = (μ^2)-(4*Δ^3)
    if L < 0 
        verbose ? println("L < 0") : nothing
        z = (μ + sqrt(-L)*im) #2z = ...,  but we only use the angle for z
        r2 = cbrt((μ^2 - L)/4)
        if r2 ≈ Δ #This must be handled more carefully!
            verbose ? println("r2 = Δ, difference: $(r2-Δ)") : nothing

            (s, c) = sincos(angle(z)/3) #can be optimized/altered to use the 'tan-formula' for the 3xReal root cubic
            k = 2*sqrt(r2)
            return (b .+ (k.* (c, (-c + (sqrt(3)*s))/2, (-c - (sqrt(3)*s))/2))) ./(-3)
        else
            verbose ? println("r2 ≠ Δ, difference: $(r2-Δ)") : nothing
            if μ ≤ 0
                verbose ? println("μ ≤ 0") : nothing
                return (b + sqrt(r2)*(1+(Δ/(r2))),) ./(-3)
            else
                verbose ? println("μ > 0") : nothing
                return (b - sqrt(r2)*(1+(Δ/(r2))),)./(-3)
            end
        end
    elseif L ≥ 0
        verbose ? println("L ≥ 0, L: $L") : nothing
        root = sqrt(L)
        if μ > 0
            z = (μ + root)/2
        else
            z = (μ - root)/2
        end
            
        C = cbrt(z)
        r2 = C^2
        if r2 ≈ Δ #This must be handled more carefully! We should check roots (amounts to a single computation) 
            verbose ? println("r2 = Δ, difference: $(r2-Δ)") : nothing
            return (b+2*C, b-C) ./ (-3)
        else
            verbose ? println("r2 ≠ Δ, difference: $(r2-Δ)") : nothing
            return (b + C*(1+(Δ/r2)),) ./(-3)
        end
    end
end

cubic_real_roots (generic function with 2 methods)

Thus we have established a pretty decent root-finder. Let's look at its performance:

In [3]:
bm_roots_1R = @benchmark cubic_real_roots($2.0, $(-3.0), $9.0) #One real root

BenchmarkTools.Trial: 10000 samples with 987 evaluations per sample.
 Range (min … max):  55.117 ns … 355.724 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     58.460 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   61.803 ns ±  12.662 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▅▆▅█▅▂▁   ▁▁▃▃▁        ▁       ▁▁▁                           ▁
  ███████████████▇█▇█▇▇▇███▇▆▆▇█████▇▆▅▄▆▇▇▇▅▆▅▆▅▆▄▅▅▅▅▄▅▄▄▄▄▄ █
  55.1 ns       Histogram: log(frequency) by time       107 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [4]:
bm_roots_3R = @benchmark cubic_real_roots($(-7.0), $(3.0), $9.0) #Three real roots

BenchmarkTools.Trial: 10000 samples with 968 evaluations per sample.
 Range (min … max):  83.781 ns …  2.040 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     88.533 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   99.631 ns ± 32.265 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▃▆█▃▁▂▃▄▃▃ ▁▂▂                 ▁▂▃▂▁▁                       ▁
  ███████████████████▇▇▆▇▆▇▆▅▇▆▇████████▇▆▇▆▆▇▅▆▆▅▅▅▅▅▄▄▅▄▄▄▂ █
  83.8 ns      Histogram: log(frequency) by time       186 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

This is something like $\sim 10-15$ times faster than the one defined above. As a rough measure of performance then we expect something like $\sim 80 N ns$ to compute the partition of the time interval for a velocity update. For e.g. the 20-dimensional Twin Peaks scenario $N = 21$ so we get something akin to $\sim 1.6 \mu s$ of work. Of course, if we wanted to we could parallellize this particular task, but the overhead for such an endeavour would probably be prohibitive. 

We need to handle some special cases as well: When the coefficients are zero (possibly some analysis of tolerance would be useful here) we get somewhat simpler equations to handle.

In [6]:
function quadratic_real_roots(c, d)
    Δ = c^2 - 4*d
    if Δ < 0
        return ()
    end
    s = sqrt(Δ)
    return ((-c-s)/2.0, (-c + s)/2.0 )
end

function triv_root(d)
    if iszero(d)
        warn("0-polynomial rate")
    end
    return ()
end

linear_real_roots(c, d) = iszero(c) ? triv_root(d) : -d/c

quadratic_real_roots(b,c,d) = iszero(b) ? linear_real_roots(c, d) : quadratic_real_roots(c/b, d/b)

cubic_real_roots(a,b,c,d) = iszero(a) ? quadratic_real_roots(b,c,d) : cubic_real_roots(b/a, c/a, d/a)

cubic_real_roots (generic function with 2 methods)

#### Partitioning the time intervals

In [8]:
cubic_real_roots(X::SVector{4, Float64}) = cubic_real_roots(X[1], X[2], X[3], X[4])
cubic_real_roots(X::AbstractArray) = cubic_real_roots(X[1], X[2], X[3], X[4])

cubic_real_roots (generic function with 4 methods)

In [ ]:
t(u, a, b, κ, start_time) = isreal(κ) ? atan((a*u + (b/2))/κ) - start_time : t(u) = atanh((a*u + (b/2))/imag(κ)) - start_time

function partition_by_roots!(T::BinaryMinHeap{Float64}, J::Integer, ρJ::Array{Float64, 2}, a, b, κ; start_time = 0.0) #could be $MVector
    empty!(T)
    #push!(T, start_time)

    for I in axes(ρJ, 1)
        if I ≠ J
            ρJI = @view(ρJ[I,:])
            roots = cubic_real_roots(ρJI)
            for root in roots
                @show root
                #The root corresponds to a velocity for which the rate becomes positive.
                 #We transform it into a time.
                root_time = t(root, a, b, κ, start_time)  
                if root_time > start_time
                    push!(T, root_time)
                end
            end
        end
    end
    return T
end

partition_by_roots! (generic function with 1 method)

In [38]:
D = 21
ρ = rand(21, 4);
T = BinaryMinHeap{Float64}()
a = rand()
b = rand()
κ = rand()
T = partition_by_roots!(T, 1, ρ, a, b, κ)

UndefVarError: UndefVarError: `partition_by_roots!` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
@benchmark partition_by_roots!($T, $1, $ρ, $a, $b, $κ)

We get roughly to $2\mu s$ of computation time, indicating that the transformation of each root takes a slight amount of time.

#### The rate integrals 
The rates need to be integrated, that is we integrate

$\lambda = A u^3 + B u^2 + C u + D = \bar{A} \cdot \bar{U}$

over time (on the intervals discussed above). Each integral

$M_n = \int u(t)^n dt$

can be analytically computed. We expand in the $\tan(\kappa (t+t_0))$ after a transformation $s = \kappa(t+t_0)$ so that

$dt = ds/\kappa$

and (by abuse of notation)

$u(s) = (\kappa \tan(s) -(b/2))/a$

whence

$M_n(s) = \frac{1}{a^n\kappa} \sum_{j=0}^n \binom{n}{j} (-b/2)^{n-j} \int (\kappa \tan(s))^{j}ds = \frac{1}{a^n\kappa} \sum_{j=0}^n \binom{n}{j} \kappa^{j}(-b/2)^{n-j} L_{j} = \frac{1}{a^n\kappa}\bar{M_n} \cdot \bar{L}$

with each $L_j$ corresponding to a $\tan(x)^j$ integral. It can be shown that

1) $L_0 = s$
2) $L_1 = -\log(\cos(s))$
3) $L_2 = (\tan(s)-s)$
3) $L_3 = (\frac{1}{2\cos^2(s)} + \log(\cos(s)))$

Obviously there are some issues with the values - they can be divergent, complex and otherwise problematic. However, we shall always have $s$ in specific domains, and the integrals shall only be integrated over domains for which $\lambda \geq 0$.

Of course we shall typically evaluate this at many distinct "times" $s$, which alters the vector $L$. Thus it makes sense to use a matrix $M$ with the distinct $M_i$ as rows so that $\bar{U}(s) = M \bar{L}(s)$

In [ ]:
function L_tuple(s::Float64)
    c = cos(s)
    lc = log(c)
    return (s, -lc, tan(s) -s , lc + (1. /(2*(c^2))))
end

In [ ]:
function M_matrix!(M::MMatrix{4,4, Float64, 16}, a, b, κ)
    for i in 1:4
        for j in 1:i
            M[i, j] = binomial(i, j) * ((-b / 2.)^j) * (κ^(i-j))
        end
        factor = (κ * (a^i))
        @views M[i,:] ./= factor
    end
    return M 
end

In [ ]:
M = @MMatrix zeros(4,4)
M_matrix!(M, rand(), randn(), rand())

In [ ]:
@benchmark M_matrix!($M, $a, $b, $κ)

In [ ]:
function compute_rate_integrals(ρ_IJs::Vector{SVector{4, Float64}}, M::MMatrix, initial_values::Vector{Float64}, s_final::Float64)
    